# Fine-tune Qwen3-0.6B on IMDB
- Types of fine-tunining
  - Instruction fine-tuning
  - Classification fine-tuning (WHAT WE DOING TODAY)
  - Preference fine-tuning

### Setup
- Setup and imports
- Configuration
- Device selection
- Load IMDB dataset
- Small train/test subsets
- Tokenizer and model
- Parameter counts and memory
- Tokenization
- Data collator, training args, and trainer
- Train, evaluate, perplexity
- Save and push to Hub
- Test generation

```bash
uv init --python 3.12
uv venv
uv add torch torchvision torchaudio transformers datasets accelerate huggingface_hub
export HF_TOKEN=...
```

In [ ]:
import os
import math
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
import transformers
import platform

print(platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

### Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "./qwen3-imdb"
MAX_LENGTH = 512
TRAIN_SAMPLES = 1000
TEST_SAMPLES = 200
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

### Device

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA available:", torch.cuda.is_available())
else:
    device = torch.device("cpu")

print("=" * 60)
print("DEVICE")
print("=" * 60)
print("Using device:", device)
print()

### Load dataset

In [ ]:
print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)
print()

### Create a small dataset for experimentation

In [ ]:
train_dataset = dataset["train"].select(range(TRAIN_SAMPLES))

test_dataset = dataset["test"].select(range(TEST_SAMPLES))

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))
print()

### Load tokenizer

In [ ]:
print("=" * 60)
print("LOADING TOKENIZER")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded.")
print()

### Load model

In [ ]:
print("=" * 60)
print("LOADING MODEL")
print("=" * 60)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    # Important for Apple Silicon MPS.
    # Avoid bfloat16.
    dtype=torch.float32,
)

model = model.to(device)

print("Model loaded.")
print()

### Model parameters

In [ ]:
print("=" * 60)
print("MODEL PARAMETERS")
print("=" * 60)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

non_trainable_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable:        {non_trainable_params:,}")
print(f"Trainable percentage: " f"{100 * trainable_params / total_params:.2f}%")
print()

### Model memory

In [ ]:
parameter_memory = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Parameter memory: " f"{parameter_memory / 1024**3:.2f} GB")
print()

### Tokenization

In [ ]:
print("=" * 60)
print("TOKENIZING DATASET")
print("=" * 60)


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)


tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)


print("Tokenization complete.")
print()
print("Example:")
print(tokenized_train[0])
print()

### Data collator

In [ ]:
print("=" * 60)
print("CREATING DATA COLLATOR")
print("=" * 60)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    # Qwen3 is a causal language model.
    mlm=False,
)

print("Data collator created.")
print()

### Training arguments

In [ ]:
print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # -----------------------------
    # Training
    # -----------------------------
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=(GRADIENT_ACCUMULATION_STEPS),
    learning_rate=LEARNING_RATE,
    # -----------------------------
    # Logging
    # -----------------------------
    logging_steps=LOGGING_STEPS,
    # -----------------------------
    # Evaluation
    # -----------------------------
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    # -----------------------------
    # Saving
    # -----------------------------
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    # -----------------------------
    # Apple Silicon / MPS
    # -----------------------------
    # fp16=False,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_pin_memory=False,
    # -----------------------------
    # Reporting
    # -----------------------------
    # report_to=["tensorboard", "wandb", "trackio", "mlflow", "none"], # "tensorboard", "wandb", "trackio", "mlflow", "none"
    # push_to_hub=True,
    # hub_model_id="worldboss/qwen3-0.6B-finetune",
)

print(training_args)
print()

### Trainer

In [ ]:
print("=" * 60)
print("CREATING TRAINER")
print("=" * 60)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer created.")
print()

### Train

In [ ]:
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

trainer.train() # train model
print()
print("Training complete.")
print()

### Evaluation

In [ ]:
print("=" * 60)
print("EVALUATING MODEL")
print("=" * 60)

results = trainer.evaluate()

print("Evaluation results:")
for key, value in results.items():
    print(f"{key}: {value}")

### Perplexity
Perplexity (PPL) is a `metric` used to measure how well a language model predicts the next token.

A simple way to think about it:
- Perplexity tells us how "surprised" the model is by the actual text.
- Lower perplexity is generally better.

Suppose the model sees:
- "The cat is sitting on the ___" A: mat (0.70), B: floor (0.15), C: chair (0.05), D: (0.01)
- f the actual next token is `mat`, the model assigned it a high probability → `low surprise` → good prediction → lower perplexity.
- If the actual token is `car`, the model assigned it a very low probability → `high surprise` → worse prediction → higher perplexity.

Mathematical definition

For a sequence of tokens:

$$ x_1,x_2,\ldots,x_N $$

perplexity is:

$$ \boxed{ PPL = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(x_i|x_{<i})\right) } $$

In words:

Perplexity is the exponential of the average negative log-likelihood of the actual tokens.

The quantity inside the exponential is essentially the model's average cross-entropy loss.

Therefore:

$$ PPL=e^{Cross Entropy Loss} $$
	

Why Lower is Better?
- Suppose you evaluate two models on the same dataset:
```
| Model   | Loss | Perplexity |
| ------- | ---: | ---------: |
| Model A | 1.20 |       3.32 |
| Model B | 2.00 |       7.39 |
```
Model A assigns higher probability to the correct tokens on average.

Therefore:
- `Model A` is better according to perplexity

```python
import math

loss = 1.20
perplexity = math.exp(loss)

print(perplexity)
# 3.320116922736547
```

Perplexity range

For standard language-model perplexity:

$$ \boxed{1 \leq PPL < \infty} $$
- PPL = 1 → perfect prediction.
- PPL close to 1 → very confident and accurate predictions.
- Higher PPL → more uncertainty / poorer predictions.
- There is no maximum value.

```
| Loss | Perplexity | Interpretation      |
| ---: | ---------: | ------------------- |
|    0 |       1.00 | Perfect             |
|  0.5 |       1.65 | Very good           |
|  1.0 |       2.72 | Good                |
|  2.0 |       7.39 | More uncertain      |
|  3.0 |      20.09 | Quite uncertain     |
|  5.0 |     148.41 | Very uncertain      |
| 10.0 |     22,026 | Extremely uncertain |
```

For LLM fine-tuning, the most useful approach is usually to compare the same model before vs. after fine-tuning on the same evaluation dataset:

$$ PPL_{before} > PPL_{after} $$
	​


Suppose our sequence is:

$$
\boxed{\text{I like cats}}
$$

The tokens are:

$$
x_1=\text{I},\quad x_2=\text{like},\quad x_3=\text{cats}
$$

The model predicts each token based on the tokens that came before it.

### 1. Assume the model gives these probabilities

Suppose the model predicts:

$$
P(\text{I}) = 0.5
$$

$$
P(\text{like}\mid\text{I}) = 0.8
$$

$$
P(\text{cats}\mid\text{I like}) = 0.25
$$

So the probability of the entire sequence is:

$$
P(\text{I like cats})
=
P(\text{I})
P(\text{like}\mid\text{I})
P(\text{cats}\mid\text{I like})
$$

Therefore:

$$
=0.5\times0.8\times0.25
$$

$$
=0.1
$$

---

### 2. Apply the perplexity formula

The mathematical definition is:

$$
PPL=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(x_i|x_{<i})
\right)
$$

We have **3 tokens**, so \(N=3\).

Substitute our probabilities:

$$
PPL=
\exp\left[
-\frac{1}{3}
\left(
\log(0.5)+
\log(0.8)+
\log(0.25)
\right)
\right]
$$

Using natural logarithms:

$$
\log(0.5)=-0.6931
$$

$$
\log(0.8)=-0.2231
$$

$$
\log(0.25)=-1.3863
$$

Therefore:

$$
PPL=
\exp\left[
-\frac{1}{3}
(-0.6931-0.2231-1.3863)
\right]
$$

$$
=\exp\left(\frac{2.3025}{3}\right)
$$

$$
=\exp(0.7675)
$$

$$
\boxed{PPL\approx2.15}
$$

### 3. What does 2.15 mean?

The model has a perplexity of approximately:

$$
\boxed{2.15}
$$

This indicates relatively low uncertainty for this particular sequence.

The model assigned fairly high probability to **"I"** and **"like"**, but was less confident about **"cats"**.

---

### The shortcut

Notice that:

$$
0.5\times0.8\times0.25=0.1
$$

We can calculate perplexity directly from the probability of the entire sequence:

$$
PPL=
\left(\frac{1}{P(x_1,\ldots,x_N)}\right)^{1/N}
$$

So:

$$
PPL=
\left(\frac{1}{0.1}\right)^{1/3}
$$

$$
=10^{1/3}
$$

$$
\boxed{PPL\approx2.15}
$$

So both methods give the same answer.

**Key intuition:**

> The model assigns probability to the correct tokens. Perplexity takes those probabilities, computes their average log-loss, and converts it back into an intuitive "effective number of choices."

For a perfectly predicted sequence, where every correct token has probability \(1\):

$$
PPL=1
$$

So **1 is perfect, and higher values indicate greater uncertainty.**


In [ ]:
if "eval_loss" in results:
    try:
        perplexity = math.exp(results["eval_loss"])
        print()
        print(f"Perplexity: {perplexity:.2f}")
    except OverflowError:
        print("Perplexity is too large to calculate.")

print()

### Save model

In [ ]:
print("=" * 60)
print("SAVING MODEL")
print("=" * 60)


FINAL_MODEL_DIR = "./qwen3-imdb-final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")
print()

# Upload model
model.push_to_hub(f"worldboss/{FINAL_MODEL_DIR}")
tokenizer.push_to_hub(f"worldboss/{FINAL_MODEL_DIR}")

### Test generation

In [ ]:
print("=" * 60)
print("TESTING FINE-TUNED MODEL")
print("=" * 60)


prompt = "This movie was absolutely fantastic because"
inputs = tokenizer(prompt, return_tensors="pt")

# Move input tensors to MPS
inputs = {key: value.to(device) for key, value in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
    )


generated_text = tokenizer.decode(outputs[0],skip_special_tokens=True,)


print()
print("Prompt:")
print(prompt)
print()
print("Generated text:")
print(generated_text)
print()
print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)